In [84]:
import librosa
import librosa.display
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import os
import pandas as pd
from transformers import ASTFeatureExtractor, ASTModel
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.metrics import roc_auc_score
from scipy.optimize import minimize

In [2]:
def build_mimii_pump_dataframe(root_dir):
    records = []

    for machine_id in os.listdir(root_dir):
        machine_path = os.path.join(root_dir, machine_id)

        if not os.path.isdir(machine_path):
            continue

        for condition in ["normal", "abnormal"]:
            condition_path = os.path.join(machine_path, condition)

            if not os.path.exists(condition_path):
                continue

            for file in os.listdir(condition_path):
                if file.endswith(".wav"):
                    records.append({
                        "machine_id": machine_id,
                        "condition": condition,
                        "file_path": os.path.join(condition_path, file)
                    })

    return pd.DataFrame(records)

In [3]:
root_dir = r"data\0_dB_pump\pump"
df = build_mimii_pump_dataframe(root_dir)

print(df.head())
print(df["condition"].value_counts())
print(df["machine_id"].value_counts())

  machine_id condition                                      file_path
0      id_00    normal  data\0_dB_pump\pump\id_00\normal\00000000.wav
1      id_00    normal  data\0_dB_pump\pump\id_00\normal\00000001.wav
2      id_00    normal  data\0_dB_pump\pump\id_00\normal\00000002.wav
3      id_00    normal  data\0_dB_pump\pump\id_00\normal\00000003.wav
4      id_00    normal  data\0_dB_pump\pump\id_00\normal\00000004.wav
condition
normal      3749
abnormal     456
Name: count, dtype: int64
machine_id
id_00    1149
id_06    1138
id_02    1116
id_04     802
Name: count, dtype: int64


In [4]:
df["label"] = df["condition"].map({"normal": 0, "abnormal": 1})

In [5]:
normal_df=df[df['condition']=='normal']
abnormal_df = df[df['condition']=='abnormal']
train_df = normal_df.iloc[:int(len(normal_df)*0.7)]
temp_df = pd.concat([normal_df.iloc[int(len(normal_df)*0.7):], abnormal_df])
val_df, test_df   = train_test_split(temp_df, test_size=0.5, shuffle=True)

In [6]:
print("Validation Data:",val_df['condition'].value_counts())
print("Training Data:",train_df['condition'].value_counts())
print("Test Data:",test_df['condition'].value_counts())

Validation Data: condition
normal      561
abnormal    229
Name: count, dtype: int64
Training Data: condition
normal    2624
Name: count, dtype: int64
Test Data: condition
normal      564
abnormal    227
Name: count, dtype: int64


In [7]:
y_val = val_df['label'].values
y_test = test_df['label'].values

In [8]:


class DeepSVDD_Network(nn.Module):
    """
    Maps AST embeddings to a compact hypersphere latent space.
    No bias terms, no batch norm with learnable affine — prevents hypersphere collapse.
    """
    def __init__(self, input_dim: int = 768, hidden_dims: list = [512, 256, 128], rep_dim: int = 128):
        super().__init__()
        layers = []
        in_dim = input_dim
        for h_dim in hidden_dims:
            layers += [
                nn.Linear(in_dim, h_dim, bias=False),
                nn.BatchNorm1d(h_dim, affine=False),  # affine=False is critical
                nn.ReLU()
            ]
            in_dim = h_dim
        layers.append(nn.Linear(in_dim, rep_dim, bias=False))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

In [91]:
class DeepSVDD:
    def __init__(self, rep_dim=128, nu=0.1, device="cuda"):
        """
        nu : float in (0,1] — upper bound on fraction of anomalies (soft-boundary)
        """
        self.rep_dim = rep_dim
        self.nu = nu
        self.device = device
        self.center = None
        self.radius = 0.0
        self.model = None

    def initialize_center(self, loader, model, eps=0.1):
        model.eval()
        all_z = []
        with torch.no_grad():
            for batch in loader:
                x = batch[0] if isinstance(batch, (list, tuple)) else batch["input_values"]
                x = x.to(self.device)
                z = model(x)
                all_z.append(z)
        c = torch.cat(all_z).mean(dim=0)
        c[(torch.abs(c) < eps) & (c < 0)] = -eps
        c[(torch.abs(c) < eps) & (c >= 0)] = eps
        return c

    def train(self, train_loader, val_loader, input_dim=768, hidden_dims=[512, 256],
          n_epochs=50, lr=1e-4, weight_decay=1e-6, warm_up_epochs=10):

        if self.model is None:
            self.model = DeepSVDD_Network(input_dim, hidden_dims, self.rep_dim).to(self.device)

        optimizer = torch.optim.Adam(self.model.parameters(), lr=lr, weight_decay=weight_decay)
        scheduler = torch.optim.lr_scheduler.MultiStepLR(
            optimizer, milestones=[int(n_epochs * 0.5), int(n_epochs * 0.75)], gamma=0.1
        )

        self.center = None
        best_val_auc = 0.0
        best_weights = None
        alpha = None   # ← CD-SVDD: dual variables

        for epoch in range(n_epochs):

            if epoch == warm_up_epochs:
                print(f"Initializing center at epoch {epoch} via dual solve...")
                # # Get all embeddings to solve dual
                # self.model.eval()
                # all_z = []
                # with torch.no_grad():
                #     for x, _ in train_loader:
                #         all_z.append(self.model(x.to(self.device)).cpu().numpy())
                # phi_x = np.vstack(all_z)

                # # CD-SVDD Step 1: solve dual → exact c and R
                # alpha = solve_dual(phi_x, self.nu)
                # c_np, R_bar = compute_center_radius(phi_x, alpha, self.nu)
                # self.center = torch.FloatTensor(c_np).to(self.device)
                # self.radius = R_bar
                ## batchwise
                x_init, _ = next(iter(train_loader))
                with torch.no_grad():
                    z_init = self.model(x_init.to(self.device))
                phi_init = z_init.cpu().numpy()
                alpha_init = solve_dual_minibatch(phi_init, self.nu)
                c_np, R_bar = compute_center_radius(phi_init, alpha_init, self.nu)
                self.center = torch.FloatTensor(c_np).to(self.device)
                self.radius = R_bar

            # ── Training ──────────────────────────────────────────────
            self.model.train()
            train_loss = 0.0

            for x, _ in train_loader:
                x = x.to(self.device)
                optimizer.zero_grad()
                z = self.model(x)

                if self.center is None:
                    loss = torch.mean(torch.sum(z ** 2, dim=1))  # warm-up
                else:
                    # CD-SVDD Step 2: update θ with exact c and R
                    ## For whole dataset
                    # loss = cd_svdd_loss(z, c_np, R_bar, alpha,
                    #                     self.nu, 1e-6, self.model, self.device)
                    # ── CD-SVDD mini-batch dual solve ──────────────
                    phi_x_batch = z.detach().cpu().numpy()          # detach — dual is solved outside graph
                    alpha_batch  = solve_dual_minibatch(phi_x_batch, self.nu)
                    c_np, R_bar  = compute_center_radius(phi_x_batch, alpha_batch, self.nu)

                    # Update global center and radius with EMA for stability
                    c_new = torch.FloatTensor(c_np).to(self.device)
                    self.center = 0.9 * self.center + 0.1 * c_new   # EMA smoothing
                    self.radius  = 0.9 * self.radius + 0.1 * R_bar
                    
                    loss = cd_svdd_loss(z, self.center.cpu().numpy(), self.radius,
                            alpha_batch, self.nu, 1e-6, self.model, self.device)

                loss.backward()
                optimizer.step()
                train_loss += loss.item()

            # CD-SVDD: re-solve dual after each epoch (alternating algorithm)
            # if self.center is not None and epoch > warm_up_epochs:
            #     self.model.eval()
            #     all_z = []
            #     with torch.no_grad():
            #         for x, _ in train_loader:
            #             all_z.append(self.model(x.to(self.device)).cpu().numpy())
            #     phi_x = np.vstack(all_z)
            #     alpha = solve_dual(phi_x, self.nu)
            #     c_np, R_bar = compute_center_radius(phi_x, alpha, self.nu)
            #     self.center = torch.FloatTensor(c_np).to(self.device)
            #     self.radius = R_bar

            scheduler.step()

            # ── Validation ────────────────────────────────────────────
            if self.center is None:
                continue

            val_scores, val_labels = self.predict(val_loader)
            val_auc = roc_auc_score(val_labels, val_scores)

            if val_auc > best_val_auc:
                best_val_auc = val_auc
                best_weights = {k: v.clone() for k, v in self.model.state_dict().items()}

            if (epoch + 1) % 10 == 0:
                print(f"Epoch [{epoch+1}/{n_epochs}] "
                    f"Train Loss: {train_loss/len(train_loader):.6f} | "
                    f"Val AUC: {val_auc:.4f} | "
                    f"Best Val AUC: {best_val_auc:.4f} | "
                    f"R: {self.radius:.4f}")

        self.model.load_state_dict(best_weights)
        print(f"\nTraining complete. Best Val AUC: {best_val_auc:.4f}")

    def _update_radius(self, dist):
        """Radius = (1-nu) quantile of distances."""
        return torch.quantile(torch.sqrt(dist).detach(), 1 - self.nu).item()

    @torch.no_grad()
    def predict(self, loader):
        """Returns anomaly scores (distance from center). Higher = more anomalous."""
        self.model.eval()
        scores, labels = [], []
        for x, y in loader:
            x = x.to(self.device)
            z = self.model(x)
            dist = torch.sum((z - self.center) ** 2, dim=1)
            scores.extend(dist.cpu().numpy())
            labels.extend(y.numpy())
        return np.array(scores), np.array(labels)
    @torch.no_grad()
    def predict_unlabeled(self, loader, threshold=None):
        """
        For unseen, unlabeled data.
        Returns anomaly scores and binary predictions (0=normal, 1=anomaly).
        """
        self.model.eval()
        scores = []

        for batch in loader:
            x = batch[0] if isinstance(batch, (list, tuple)) else batch["input_values"]
            x = x.to(self.device)
            z = self.model(x)
            dist = torch.sum((z - self.center) ** 2, dim=1)
            scores.extend(dist.cpu().numpy())

        scores = np.array(scores)

        # Use radius as threshold if not provided
        threshold = threshold or self.radius
        preds = (scores > threshold).astype(int)  # 1 = anomaly, 0 = normal

        return scores, preds

In [ ]:
def solve_dual(phi_x, nu):
    """
    Solve dual QP (from CD-SVDD paper).
    phi_x: np.ndarray of shape (l, rep_dim)
    Returns: alpha* of shape (l,)
    """
    l = phi_x.shape[0]
    Q = phi_x @ phi_x.T
    Q_diag = np.diag(Q)
    upper_bound = 1.0 / (nu * l)

    def objective(alpha):
        return alpha @ Q @ alpha - alpha @ Q_diag

    def grad(alpha):
        return 2 * Q @ alpha - Q_diag

    constraints = {"type": "eq", "fun": lambda a: np.sum(a) - 1}
    bounds = [(0, upper_bound)] * l
    alpha0 = np.ones(l) / l

    result = minimize(objective, alpha0, jac=grad,
                      method="SLSQP", bounds=bounds,
                      constraints=constraints,
                      options={"maxiter": 500, "ftol": 1e-9})
    return result.x

def solve_dual_minibatch(phi_x_batch, nu):
    """
    Solve dual QP on a single mini-batch instead of full dataset.
    phi_x_batch: np.ndarray (batch_size, rep_dim)
    """
    l = phi_x_batch.shape[0]
    Q = phi_x_batch @ phi_x_batch.T
    Q_diag = np.diag(Q)
    upper_bound = 1.0 / (nu * l)

    def objective(alpha):
        return alpha @ Q @ alpha - alpha @ Q_diag

    def grad(alpha):
        return 2 * Q @ alpha - Q_diag

    constraints = {"type": "eq", "fun": lambda a: np.sum(a) - 1}
    bounds = [(0, upper_bound)] * l
    alpha0 = np.ones(l) / l

    result = minimize(objective, alpha0, jac=grad,
                      method="SLSQP", bounds=bounds,
                      constraints=constraints,
                      options={"maxiter": 200, "ftol": 1e-7})
    return result.x
    
def compute_center_radius(phi_x, alpha, nu):
    """
    Exact c* (Eq. 21) and R̄* (Eq. 22) from CD-SVDD paper.
    phi_x: np.ndarray (l, rep_dim)
    alpha: np.ndarray (l,)
    """
    l = phi_x.shape[0]
    upper_bound = 1.0 / (nu * l)
    eps = 1e-6

    # Eq. 21: exact center
    c = (alpha[:, None] * phi_x).sum(axis=0)

    # Eq. 22: boundary SVs → 0 < αᵢ < 1/(νl)
    sv_mask = (alpha > eps) & (alpha < upper_bound - eps)
    if sv_mask.sum() == 0:
        sv_mask = alpha > eps   # fallback

    dists = np.sum((phi_x[sv_mask] - c) ** 2, axis=1)
    R_bar = float(dists.mean())

    return c, R_bar


def cd_svdd_loss(z, c_np, R_bar, alpha_np, nu, lambda_reg, model, device):
    """
    CD-SVDD loss function (Eq. 23).
    z      : torch tensor (batch, rep_dim) — network output
    c_np   : np.ndarray (rep_dim,) — exact center
    R_bar  : float — exact radius squared
    """
    l = z.shape[0]
    upper_bound = 1.0 / (nu * l)
    eps = 1e-6

    c = torch.FloatTensor(c_np).to(device)
    R_bar_t = torch.tensor(R_bar, dtype=torch.float32).to(device)

    sv_mask = (alpha_np > eps) & (alpha_np < upper_bound - eps)
    if sv_mask.sum() == 0:
        sv_mask = alpha_np > eps

    # Only use boundary SVs that fall within current batch size
    sv_idx = torch.BoolTensor(sv_mask[:l]).to(device)

    # Term 1: mean dist of boundary SVs
    if sv_idx.sum() > 0:
        term1 = torch.mean(torch.sum((z[sv_idx] - c) ** 2, dim=1))
    else:
        term1 = torch.tensor(0.0).to(device)

    # Term 2: penalty for points outside hypersphere
    dist_all = torch.sum((z - c) ** 2, dim=1)
    term2 = (1.0 / (nu * l)) * torch.mean(torch.clamp(dist_all - R_bar_t, min=0))

    # Regularization
    reg = sum(torch.norm(p) for p in model.parameters())

    return term1 + term2 + (1e-6 / 2) * reg

In [10]:
class DeepSVDD_Autoencoder(nn.Module):
    def __init__(self, input_dim=1536, hidden_dims=[512, 256], rep_dim=128):
        super().__init__()

        # Encoder (same architecture as SVDD network)
        enc_layers = []
        in_dim = input_dim
        for h in hidden_dims:
            enc_layers += [nn.Linear(in_dim, h, bias=False),
                           nn.BatchNorm1d(h, affine=False),
                           nn.ReLU()]
            in_dim = h
        enc_layers.append(nn.Linear(in_dim, rep_dim, bias=False))
        self.encoder = nn.Sequential(*enc_layers)

        # Decoder (mirror of encoder)
        dec_layers = []
        in_dim = rep_dim
        for h in reversed(hidden_dims):
            dec_layers += [nn.Linear(in_dim, h, bias=False),
                           nn.BatchNorm1d(h, affine=False),
                           nn.ReLU()]
            in_dim = h
        dec_layers.append(nn.Linear(in_dim, input_dim, bias=False))
        self.decoder = nn.Sequential(*dec_layers)

    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat, z


def pretrain_autoencoder(train_loader, input_dim=1536, hidden_dims=[512, 256],
                         rep_dim=128, n_epochs=50, lr=1e-4, device="cuda"):
    ae = DeepSVDD_Autoencoder(input_dim, hidden_dims, rep_dim).to(device)
    optimizer = torch.optim.Adam(ae.parameters(), lr=lr, weight_decay=1e-6)
    criterion = nn.MSELoss()

    for epoch in range(n_epochs):
        ae.train()
        epoch_loss = 0.0
        for x, _ in train_loader:
            x = x.to(device)
            optimizer.zero_grad()
            x_hat, _ = ae(x)
            loss = criterion(x_hat, x)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        if (epoch + 1) % 10 == 0:
            print(f"AE Epoch [{epoch+1}/{n_epochs}] Loss: {epoch_loss/len(train_loader):.6f}")

    return ae

In [23]:
def build_svdd_from_ae(ae, input_dim=1536, hidden_dims=[512, 256], rep_dim=128):
    svdd_net = DeepSVDD_Network(input_dim, hidden_dims, rep_dim)

    # Copy encoder weights layer by layer
    ae_enc_state   = ae.encoder.state_dict()
    svdd_net_state = svdd_net.net.state_dict()

    for (ae_k, ae_v), (sv_k, sv_v) in zip(ae_enc_state.items(), svdd_net_state.items()):
        if ae_v.shape == sv_v.shape:
            svdd_net_state[sv_k] = ae_v.clone()

    svdd_net.net.load_state_dict(svdd_net_state)
    return svdd_net

In [12]:
feature_extractor = ASTFeatureExtractor.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593")
ast_model = ASTModel.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

ASTModel LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                         | Status     |  | 
----------------------------+------------+--+-
classifier.layernorm.bias   | UNEXPECTED |  | 
classifier.dense.weight     | UNEXPECTED |  | 
classifier.layernorm.weight | UNEXPECTED |  | 
classifier.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [13]:
class ASTFineTuner(nn.Module):
    def __init__(self, ast_model, hidden_size=768, mask_ratio=0.4):
        super().__init__()
        self.ast = ast_model
        self.mask_ratio = mask_ratio

        # Freeze all layers first
        for param in self.ast.parameters():
            param.requires_grad = False

        # Unfreeze only last 4 transformer blocks
        for layer in self.ast.encoder.layer[-4:]:
            for param in layer.parameters():
                param.requires_grad = True

        # Reconstruction head
        self.decoder = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.GELU(),
            nn.Linear(hidden_size, hidden_size)
        )

    def forward(self, input_values, mask=None):
        outputs = self.ast(input_values=input_values, output_hidden_states=True)
        patch_embeddings = outputs.last_hidden_state  # (B, num_patches, 768)

        if mask is not None:
            # Reconstruct masked patches
            reconstructed = self.decoder(patch_embeddings)
            return reconstructed, patch_embeddings, mask
        else:
            # Inference: return CLS + mean pooled embedding
            cls_emb  = patch_embeddings[:, 0, :]            # (B, 768)
            mean_emb = patch_embeddings[:, 1:, :].mean(1)   # (B, 768)
            return torch.cat([cls_emb, mean_emb], dim=1)    # (B, 1536)

    def generate_mask(self, batch_size, num_patches, device):
        num_mask = int(self.mask_ratio * num_patches)
        mask = torch.zeros(batch_size, num_patches, dtype=torch.bool, device=device)
        for i in range(batch_size):
            idx = torch.randperm(num_patches)[:num_mask]
            mask[i, idx] = True
        return mask

In [14]:
def finetune_ast(model, train_loader, val_loader, n_epochs=30, lr=1e-4, device="cuda"):
    model = model.to(device)
    
    # Only optimize unfrozen params
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=1e-4
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)
    criterion = nn.MSELoss()

    best_val_loss = float("inf")
    best_weights = None

    for epoch in range(n_epochs):
        # ── Train ──────────────────────────────────────────────
        model.train()
        train_loss = 0.0

        for batch in train_loader:
            input_values = batch["input_values"].to(device)

            # Get number of patches from a forward pass
            with torch.no_grad():
                tmp = model.ast(input_values=input_values[:1])
                num_patches = tmp.last_hidden_state.shape[1] - 1  # exclude CLS

            mask = model.generate_mask(input_values.size(0), num_patches, device)

            optimizer.zero_grad()
            reconstructed, original, mask = model(input_values, mask=mask)

            # Loss only on masked patches (skip CLS token → offset by 1)
            mask_ext = mask.unsqueeze(-1).expand_as(reconstructed[:, 1:, :])
            loss = criterion(
                reconstructed[:, 1:, :][mask_ext],
                original[:, 1:, :][mask_ext].detach()
            )

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss += loss.item()

        scheduler.step()

        # ── Validate ───────────────────────────────────────────
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch in val_loader:
                input_values = batch["input_values"].to(device)
                tmp = model.ast(input_values=input_values[:1])
                num_patches = tmp.last_hidden_state.shape[1] - 1
                mask = model.generate_mask(input_values.size(0), num_patches, device)
                reconstructed, original, mask = model(input_values, mask=mask)
                mask_ext = mask.unsqueeze(-1).expand_as(reconstructed[:, 1:, :])
                loss = criterion(
                    reconstructed[:, 1:, :][mask_ext],
                    original[:, 1:, :][mask_ext].detach()
                )
                val_loss += loss.item()

        avg_val = val_loss / len(val_loader)
        if avg_val < best_val_loss:
            best_val_loss = avg_val
            best_weights = {k: v.clone() for k, v in model.state_dict().items()}

        if (epoch + 1) % 5 == 0:
            print(f"Epoch [{epoch+1}/{n_epochs}] "
                  f"Train Loss: {train_loss/len(train_loader):.6f} | "
                  f"Val Loss: {avg_val:.6f} | "
                  f"Best: {best_val_loss:.6f}")

    model.load_state_dict(best_weights)
    return model

In [15]:
@torch.no_grad()
def extract_finetuned_embeddings(model, audio_list, feature_extractor, device="cuda"):
    """
    audio_list : list of raw waveforms (np.ndarray, sr=16000)
    Returns    : np.ndarray of shape (N, 1536)
    """
    model.eval()
    all_embeddings = []

    for waveform in audio_list:
        inputs = feature_extractor(
            waveform,
            sampling_rate=16000,
            return_tensors="pt"
        )
        input_values = inputs["input_values"].to(device)
        emb = model(input_values)          # (1, 1536)
        all_embeddings.append(emb.cpu().numpy())

    return np.vstack(all_embeddings)

In [16]:
def extract_waveforms(df):
    waveforms=[]   
    for file in df['file_path']: 
        waveform, sr = librosa.load(file, sr=16000)
        waveforms.append(waveform)
    return waveforms

In [17]:
train_waveforms = extract_waveforms(train_df)
val_waveforms   = extract_waveforms(val_df)
test_waveforms  = extract_waveforms(test_df)

In [18]:
# ── Step 1: Build DataLoader ───────────────────────────────────────────
class MachineAudioDataset(torch.utils.data.Dataset):
    def __init__(self, waveforms, feature_extractor):
        self.waveforms = waveforms
        self.fe = feature_extractor

    def __len__(self):
        return len(self.waveforms)

    def __getitem__(self, idx):
        inputs = self.fe(self.waveforms[idx], sampling_rate=16000, return_tensors="pt")
        return {"input_values": inputs["input_values"].squeeze(0)}

train_dataset = MachineAudioDataset(train_waveforms, feature_extractor)
val_dataset   = MachineAudioDataset(val_waveforms,   feature_extractor)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=16, shuffle=False)

In [ ]:

# ── Step 2: Fine-tune ──────────────────────────────────────────────────
ast_finetuner = ASTFineTuner(ast_model, mask_ratio=0.4)
ast_finetuner = finetune_ast(ast_finetuner, train_loader, val_loader, n_epochs=30)


Epoch [5/30] Train Loss: 0.006095 | Val Loss: 0.000457 | Best: 0.000457
Epoch [10/30] Train Loss: 0.001326 | Val Loss: 0.001944 | Best: 0.000453
Epoch [15/30] Train Loss: 0.000662 | Val Loss: 0.001027 | Best: 0.000453
Epoch [20/30] Train Loss: 0.000461 | Val Loss: 0.000580 | Best: 0.000453
Epoch [25/30] Train Loss: 0.000685 | Val Loss: 0.000893 | Best: 0.000453
Epoch [30/30] Train Loss: 0.000681 | Val Loss: 0.000826 | Best: 0.000453


C:\Users\DEBJIT\AppData\Local\Temp\ipykernel_2740\2401493603.py:42: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  torch.FloatTensor(y_val)


In [32]:
#Save the model
save_path = "AST_model/ast_finetuned"
os.makedirs(save_path, exist_ok=True)

# Save full finetuner (includes AST + decoder head)
torch.save({
    "model_state_dict": ast_finetuner.state_dict(),
    "mask_ratio": ast_finetuner.mask_ratio,
}, f"{save_path}/overall_ast_finetuner.pt")

print(f"Model saved to {save_path}/overall_ast_finetuner.pt")

Model saved to AST_model/ast_finetuned/overall_ast_finetuner.pt


In [19]:
##To reload AST model later
# Rebuild the model architecture first
ast_model = ASTModel.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593")
ast_finetuner = ASTFineTuner(ast_model, mask_ratio=0.4)

# Load saved weights
checkpoint = torch.load("AST_model/ast_finetuned/overall_ast_finetuner.pt", map_location="cuda")
ast_finetuner.load_state_dict(checkpoint["model_state_dict"])
ast_finetuner = ast_finetuner.to("cuda")
ast_finetuner.eval()

print("Model loaded successfully")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

ASTModel LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                         | Status     |  | 
----------------------------+------------+--+-
classifier.layernorm.bias   | UNEXPECTED |  | 
classifier.dense.weight     | UNEXPECTED |  | 
classifier.layernorm.weight | UNEXPECTED |  | 
classifier.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully


In [21]:
# ── Step 3: Extract embeddings ─────────────────────────────────────────
X_train_emb = extract_finetuned_embeddings(ast_finetuner, train_waveforms, feature_extractor)
X_val_emb   = extract_finetuned_embeddings(ast_finetuner, val_waveforms,   feature_extractor)
X_test_emb  = extract_finetuned_embeddings(ast_finetuner, test_waveforms,  feature_extractor)

# ── Step 4: Feed into Deep SVDD ────────────────────────────────────────
scaler = StandardScaler()
X_train_scaled = normalize(scaler.fit_transform(X_train_emb))   # normal samples only
X_val_scaled  = normalize(scaler.transform(X_val_emb))
X_test_scaled = normalize(scaler.transform(X_test_emb))

train_tensor = TensorDataset(
    torch.FloatTensor(X_train_scaled),
    torch.zeros(len(X_train_scaled))  # dummy labels
)

val_tensor = TensorDataset(
    torch.FloatTensor(X_val_scaled),
    torch.FloatTensor(y_val)
)

test_tensor = TensorDataset(
    torch.FloatTensor(X_test_scaled)
)

train_loader_emb = DataLoader(train_tensor, batch_size=64, shuffle=True,  drop_last=True)
test_loader_emb  = DataLoader(test_tensor,  batch_size=64, shuffle=False)
val_loader_emb   = DataLoader(val_tensor,   batch_size=64, shuffle=False)


C:\Users\DEBJIT\AppData\Local\Temp\ipykernel_20872\1921918231.py:19: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  torch.FloatTensor(y_val)


In [96]:
# Stage 1 — Pretrain Autoencoder
ae = pretrain_autoencoder(train_loader_emb, input_dim=1536, hidden_dims=[256, 128],
                          rep_dim=128, n_epochs=50)



AE Epoch [10/50] Loss: 0.000829
AE Epoch [20/50] Loss: 0.000506
AE Epoch [30/50] Loss: 0.000408
AE Epoch [40/50] Loss: 0.000350
AE Epoch [50/50] Loss: 0.000330


In [93]:
# Stage 2 — Transfer weights
svdd = DeepSVDD(rep_dim=128, nu=0.2, device="cuda")
svdd.model = build_svdd_from_ae(ae).to("cuda")

# Stage 3 — Train SVDD
svdd.train(train_loader_emb, val_loader_emb, input_dim=1536, hidden_dims=[512, 256],n_epochs=100,lr=1e-2)
scores, labels = svdd.predict(val_loader_emb)
auc = roc_auc_score(labels, scores)
print(f"Val ROC-AUC: {auc:.4f}")

Initializing center at epoch 10 via dual solve...
Epoch [20/100] Train Loss: 0.064993 | Val AUC: 0.2604 | Best Val AUC: 0.7543 | R: 0.0318
Epoch [30/100] Train Loss: 0.095956 | Val AUC: 0.4295 | Best Val AUC: 0.7543 | R: 0.0402
Epoch [40/100] Train Loss: 0.149786 | Val AUC: 0.5270 | Best Val AUC: 0.7543 | R: 0.0799
Epoch [50/100] Train Loss: 1.703761 | Val AUC: 0.3306 | Best Val AUC: 0.7543 | R: 0.6093
Epoch [60/100] Train Loss: 0.003266 | Val AUC: 0.5025 | Best Val AUC: 0.7543 | R: 0.0034
Epoch [70/100] Train Loss: 0.001924 | Val AUC: 0.5025 | Best Val AUC: 0.7543 | R: 0.0018
Epoch [80/100] Train Loss: 0.001490 | Val AUC: 0.5087 | Best Val AUC: 0.7543 | R: 0.0014
Epoch [90/100] Train Loss: 0.001328 | Val AUC: 0.5043 | Best Val AUC: 0.7543 | R: 0.0013
Epoch [100/100] Train Loss: 0.001455 | Val AUC: 0.5138 | Best Val AUC: 0.7543 | R: 0.0014

Training complete. Best Val AUC: 0.7543
Val ROC-AUC: 0.7852


In [ ]:
from sklearn.metrics import roc_curve
from sklearn.metrics import accuracy_score
fpr, tpr, thresholds = roc_curve(y_val, scores)
optimal_idx = np.argmax(tpr - fpr)                    # Youden's J
optimal_threshold = thresholds[optimal_idx]
print(f"Optimal threshold from val: {optimal_threshold:.4f}")
test_scores, preds = svdd.predict_unlabeled(test_loader_emb, threshold=optimal_threshold)

# Results
y_pred=[]
for i, (s, p) in enumerate(zip(test_scores, preds)):
    print(f"Sample {i:4d} | Score: {s:.4f} | {'ANOMALY' if p else 'normal'}")

print("="*30)
accuracy_fraction = accuracy_score(y_test, preds)
print(f"Accuracy (fraction): {accuracy_fraction}")

# Calculate the number of correct predictions
accuracy_count = accuracy_score(y_test, preds, normalize=False)
print(f"Accuracy (count): {accuracy_count}")

Optimal threshold from val: 1.4454
Sample    0 | Score: 1.6267 | ANOMALY
Sample    1 | Score: 1.5135 | ANOMALY
Sample    2 | Score: 1.5468 | ANOMALY
Sample    3 | Score: 1.4297 | normal
Sample    4 | Score: 1.1970 | normal
Sample    5 | Score: 1.7918 | ANOMALY
Sample    6 | Score: 1.6245 | ANOMALY
Sample    7 | Score: 1.9070 | ANOMALY
Sample    8 | Score: 1.0449 | normal
Sample    9 | Score: 0.8206 | normal
Sample   10 | Score: 1.5824 | ANOMALY
Sample   11 | Score: 1.4247 | normal
Sample   12 | Score: 1.8571 | ANOMALY
Sample   13 | Score: 0.9054 | normal
Sample   14 | Score: 0.9177 | normal
Sample   15 | Score: 1.7028 | ANOMALY
Sample   16 | Score: 1.5845 | ANOMALY
Sample   17 | Score: 0.8222 | normal
Sample   18 | Score: 0.9543 | normal
Sample   19 | Score: 1.3010 | normal
Sample   20 | Score: 0.9823 | normal
Sample   21 | Score: 1.4358 | normal
Sample   22 | Score: 1.0797 | normal
Sample   23 | Score: 1.4893 | ANOMALY
Sample   24 | Score: 1.5446 | ANOMALY
Sample   25 | Score: 1.4795 

In [95]:
#Save the model
save_path = "SVDD_model/svdd_model"
os.makedirs(save_path, exist_ok=True)


torch.save({
    "model_state_dict": svdd.model.state_dict(),
    "center": svdd.center,
    "radius": svdd.radius,
    "rep_dim": svdd.rep_dim,
    "nu": svdd.nu,
}, f"{save_path}/overall_svdd_best.pt")

print(f"SVDD saved to {save_path}/overall_svdd_best.pt")

SVDD saved to SVDD_model/svdd_model/overall_svdd_best.pt


In [ ]:
## to reload SVDD model
# Rebuild architecture first
svdd = DeepSVDD(rep_dim=128, nu=0.1, device="cuda")
svdd.model = DeepSVDD_Network(input_dim=1536, hidden_dims=[256, 128], rep_dim=128).to("cuda")

# Load checkpoint
checkpoint = torch.load("SVDD_model/svdd_model/overall_svdd_best.pt", map_location="cuda")
svdd.model.load_state_dict(checkpoint["model_state_dict"])
svdd.center = checkpoint["center"]
svdd.radius = checkpoint["radius"]
svdd.model = svdd.model.to("cuda")
svdd.model.eval()

print("SVDD loaded successfully")

SVDD loaded successfully
